In [1]:
# Polariton Disorder — Computation Notebook

# Runs the kernel pre-computation and Picard self-energy solve for one or more
# parameter sets.  Results are saved to `Results/` as `.npy` arrays with
# accompanying `_meta.json` sidecar files.

# **Workflow**
# 1. Define a base `Params` and any sweep axes in *Cell 2 – Parameters & sweep*.
# 2. Run *Cell 3 – Sweep helpers* to expand the parameter combinations.
# 3. Run *Cell 4 – Kernel mesh* to build and save `K(q,k)` integrand meshes.
# 4. Run *Cell 5 – Picard solve* to compute `Q(k, eta)` for all sweep points.
# 5. Open `Visualisations.ipynb` to plot saved results.


In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from polaritons.parameters import Params, DEFAULT_PARAMS

# ---------------------------------------------------------------------------
# Base parameter set  (modify here to change the physics)
# ---------------------------------------------------------------------------

base = Params(
	# Material (GaAs)
	E_bind       = 4.2e-3,          # eV  -- exciton binding energy
	E_gap_bare   = 1.6,             # eV  -- bare band gap
	m_e          = 3.81e-13,        # eV s^2/m^2
	m_h          = 2.56e-12,        # eV s^2/m^2
	m_rest       = 5.68e-12,        # eV s^2/m^2

	# Polariton / cavity
	Omega        = 1.4e-2,          # eV  -- Rabi splitting
	m_prime      = 1.0,
	n_refr       = 3.0,             # cavity refractive index
	N_qw         = 1,               # number of quantum wells

	# Disorder
	D_0          = 2.26e-20,        # eV^2 m^2
	xi           = 20e-9,           # m  -- correlation length

	# Thermodynamics
	T            = 20.0,            # K
	concentration = 0.3e12,         # m^-2
	g_ex         = 12e-18,          # eV m^2
)

# ---------------------------------------------------------------------------
# Parameter sweep axes
#
# Each entry is  (param_name, list_of_values).
# All combinations are explored.  Use a single-element list to hold a param fixed.
# ---------------------------------------------------------------------------

SWEEP = {
	"xi"    : [10e-9, 20e-9, 40e-9],   # correlation length (m)
	"D_0"   : [2.26e-20],              # disorder strength -- single value keeps it fixed
}

# eta grid -- disorder amplitude sweep (always included)
eta_grid = np.round(np.linspace(0.0, 2.0, 21), 9)

print(f"Base params:\n  E_bind={base.E_bind*1e3:.1f} meV,  xi={base.xi*1e9:.0f} nm,  "
	f"T={base.T:.0f} K,  D_0={base.D_0:.2e} eV^2m^2,  m_prime={base.m_prime:g}")
print(f"Sweep axes: {list(SWEEP.keys())}")
print(f"eta_grid: {len(eta_grid)} points from {eta_grid[0]} to {eta_grid[-1]}")


Base params:
  E_bind=4.2 meV,  xi=20 nm,  T=20 K,  D_0=2.26e-20 eV^2m^2,  m_prime=1
Sweep axes: ['xi', 'D_0']
eta_grid: 21 points from 0.0 to 2.0


In [3]:
import itertools
from dataclasses import replace

def build_sweep_params(base: Params, sweep: dict) -> list[Params]:
	"""
	Return a list of Params objects, one for each combination of sweep values.

	Parameters
	----------
	base   : base Params (SI units)
	sweep  : dict mapping field names to lists of values to sweep over

	Example
	-------
	sweep = {"xi": [10e-9, 20e-9], "T": [10.0, 20.0]}
	→ 4 Params objects covering all (xi, T) combinations
	"""
	keys   = list(sweep.keys())
	values = list(sweep.values())
	combos = list(itertools.product(*values))

	params_list = []
	for combo in combos:
		overrides = dict(zip(keys, combo))
		p = replace(base, **overrides)
		params_list.append(p)
	return params_list


sweep_params_si      = build_sweep_params(base, SWEEP)
sweep_params_natural = [p.to_natural() for p in sweep_params_si]

print(f"{len(sweep_params_si)} parameter set(s) in sweep:")
for i, p in enumerate(sweep_params_si):
	print(f"  [{i}]  xi={p.xi*1e9:.0f} nm  D_0={p.D_0:.2e}  T={p.T:.0f} K")


3 parameter set(s) in sweep:
  [0]  xi=10 nm  D_0=2.26e-20  T=20 K
  [1]  xi=20 nm  D_0=2.26e-20  T=20 K
  [2]  xi=40 nm  D_0=2.26e-20  T=20 K


In [4]:
from scipy.interpolate import RectBivariateSpline
from polaritons.kernel import make_kernel_gaussian, make_kernel_nongaussian
from polaritons.grid   import build_segmented_grid
from polaritons.io     import save_result, make_sweep_stem

# ---------------------------------------------------------------------------
# Grid configuration  (shared across all kernel jobs)
# ---------------------------------------------------------------------------

SWEEP_SCHEMA = "gaussian_xi_plus_single_nongaussian_v1"

# Non-Gaussian is evaluated once using the base parameter set. Gaussian is
# evaluated for every ξ sweep point.
KERNEL_FACTORIES = {
	"nongaussian": make_kernel_nongaussian,
	"gaussian"   : make_kernel_gaussian,
}

# Segment boundaries in natural momentum units
Q_A_END = 1.25    # IR / coarse boundary
Q_B_END = 25.0    # coarse / resonance boundary
Q_C_END = 100.0   # resonance / tail boundary
K_DOMAIN_MAX = 150.0  # upper momentum limit

N_COARSE = 2_000    # coarse evaluation grid
N_PICARD = 10_000   # Picard quadrature grid

# Fraction of points per segment (same for coarse and Picard grids)
FRACS = dict(frac_A=0.25, frac_B=0.10, frac_C=0.55, frac_D=0.10, power_A=2.0)

# ---------------------------------------------------------------------------
# Kernel job helpers
# ---------------------------------------------------------------------------

kernel_jobs = []


def kernel_run_label(kernel_type, p_si, xi_independent=False):
	parts = [kernel_type]
	if not xi_independent:
		parts.append(f"ξ={p_si.xi*1e9:.0f} nm")
	parts.append(f"D_0={p_si.D_0:.2e}")
	parts.append(f"m_prime={p_si.m_prime:g}")
	return ", ".join(parts)


def build_kernel_job(p_si, p_nat, *, kernel_type, sweep_index, xi_independent=False):
	print(f"\n-- {kernel_run_label(kernel_type, p_si, xi_independent)} --")

	q_coarse, _ = build_segmented_grid(N_COARSE, Q_A_END, Q_B_END, Q_C_END, K_DOMAIN_MAX, **FRACS)
	q_picard, counts = build_segmented_grid(N_PICARD, Q_A_END, Q_B_END, Q_C_END, K_DOMAIN_MAX, **FRACS)

	print(f"  Picard grid: N={len(q_picard)}, "
		  f"N_A={counts['N_A']}, N_B={counts['N_B']}, "
		  f"N_C={counts['N_C']}, N_D={counts['N_D']}")

	K_fn = KERNEL_FACTORIES[kernel_type](p_nat, n_gauss=96)
	print(f"  Computing kernel mesh ({N_COARSE}x{N_COARSE}) ...")
	mesh_coarse = K_fn(q_coarse, q_coarse)

	interp = RectBivariateSpline(q_coarse, q_coarse, mesh_coarse)
	mesh_picard = interp(q_picard, q_picard)

	extra_meta = {
		"kernel_type"        : kernel_type,
		"xi_independent"     : bool(xi_independent),
		"N_coarse"           : N_COARSE,
		"N_picard"           : N_PICARD,
		"q_picard"           : q_picard.tolist(),
		"grid_fracs"         : FRACS,
		"q_boundaries_natural": [Q_A_END, Q_B_END, Q_C_END, K_DOMAIN_MAX],
		"sweep_index"        : sweep_index,
		"xi_m"               : None if xi_independent else p_si.xi,
		"m_prime"            : p_si.m_prime,
		"calculation_units"  : "natural",
		"sweep_schema"       : SWEEP_SCHEMA,
	}
	stem = make_sweep_stem("K", p_nat, extra={
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"sweep_schema"   : SWEEP_SCHEMA,
	})
	save_result(mesh_picard, "Results/integrand_meshes", stem, p_nat, extra_meta)
	kernel_jobs.append({
		"sweep_index"    : sweep_index,
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"stem"           : stem,
		"p_si"           : p_si,
		"p_nat"          : p_nat,
	})


# Non-Gaussian: one reference kernel, run first.
p_ng_si = base
p_ng_nat = base.to_natural()
build_kernel_job(p_ng_si, p_ng_nat, kernel_type="nongaussian", sweep_index=None, xi_independent=True)

# Gaussian: one kernel per ξ sweep point.
for idx, (p_si, p_nat) in enumerate(zip(sweep_params_si, sweep_params_natural)):
	build_kernel_job(p_si, p_nat, kernel_type="gaussian", sweep_index=idx, xi_independent=False)

print(f"\nAll kernels saved.  Jobs: {[(job['kernel_type'], job['stem']) for job in kernel_jobs]}")



-- gaussian kernel: xi=10 nm, D_0=2.26e-20, m_prime=1 --
  Picard grid: N=10000, N_A=2500, N_B=1000, N_C=5500, N_D=1000
  Computing kernel mesh (2000x2000) ...
Saved  K_06833b.npy  [10000, 10000] float64

-- gaussian kernel: xi=20 nm, D_0=2.26e-20, m_prime=1 --
  Picard grid: N=10000, N_A=2500, N_B=1000, N_C=5500, N_D=1000
  Computing kernel mesh (2000x2000) ...
Saved  K_829b3d.npy  [10000, 10000] float64

-- gaussian kernel: xi=40 nm, D_0=2.26e-20, m_prime=1 --
  Picard grid: N=10000, N_A=2500, N_B=1000, N_C=5500, N_D=1000
  Computing kernel mesh (2000x2000) ...
Saved  K_73f4f0.npy  [10000, 10000] float64

-- nongaussian kernel: xi-independent, D_0=2.26e-20, m_prime=1 --
  Picard grid: N=10000, N_A=2500, N_B=1000, N_C=5500, N_D=1000
  Computing kernel mesh (2000x2000) ...
Saved  K_eab2b3.npy  [10000, 10000] float64

All kernels saved.  Jobs: [('gaussian', 'K_06833b'), ('gaussian', 'K_829b3d'), ('gaussian', 'K_73f4f0'), ('nongaussian', 'K_eab2b3')]


In [5]:
from polaritons.solver import picard_iteration
from polaritons.kernel import make_propagator
from polaritons.grid   import trapz_weights
from polaritons.io     import load_result, save_result, make_sweep_stem

# Picard iteration settings
PICARD_TOL      = 1e-6
PICARD_MAX_ITER = 5000
PICARD_W        = 0.99    # under-relaxation factor
PICARD_VERBOSE  = True

for job in kernel_jobs:
	kernel_type = job["kernel_type"]
	k_stem = job["stem"]
	p_si = job["p_si"]
	p_nat = job["p_nat"]
	sweep_index = job["sweep_index"]
	xi_independent = job["xi_independent"]
	print(f"\n-- Picard solve, {kernel_run_label(kernel_type, p_si, xi_independent)} --")

	# Load the corresponding kernel mesh + metadata
	K_mesh, k_meta = load_result("Results/integrand_meshes", k_stem)
	q_picard       = np.array(k_meta["q_picard"])
	N_pic          = len(q_picard)
	weights        = trapz_weights(q_picard)

	# Build the propagator for this parameter set
	F_fn = make_propagator(p_nat)

	# Warm-start from a constant complex initial guess
	Q_init = (1 + 1j) * np.ones(N_pic, dtype=complex)

	Q_results = np.zeros((len(eta_grid), N_pic), dtype=complex)

	for ei, eta in enumerate(eta_grid):
		print(f"  η={eta:.3f}  ({ei+1}/{len(eta_grid)})")
		Q_conv, _ = picard_iteration(
			Q_init, q_picard, K_mesh, weights, F_fn,
			eta=eta, tol=PICARD_TOL, max_iter=PICARD_MAX_ITER,
			w=PICARD_W, verbose=PICARD_VERBOSE,
		)
		Q_results[ei] = Q_conv
		Q_init        = Q_conv.copy()   # warm-start next η

	# Save Q results
	extra_q = {
		"eta_grid"        : eta_grid.tolist(),
		"q_picard"        : q_picard.tolist(),
		"kernel_stem"     : k_stem,
		"kernel_type"     : kernel_type,
		"xi_independent"  : bool(xi_independent),
		"picard_tol"      : PICARD_TOL,
		"picard_max_iter" : PICARD_MAX_ITER,
		"picard_w"        : PICARD_W,
		"sweep_index"     : sweep_index,
		"xi_m"            : None if xi_independent else p_si.xi,
		"m_prime"         : p_si.m_prime,
		"calculation_units": "natural",
		"sweep_schema"    : SWEEP_SCHEMA,
	}
	q_stem = make_sweep_stem("Q", p_nat, extra={
		"kernel_stem"    : k_stem,
		"kernel_type"    : kernel_type,
		"xi_independent" : bool(xi_independent),
		"sweep_schema"   : SWEEP_SCHEMA,
	})
	save_result(Q_results, "Results/Q_results", q_stem, p_nat, extra_q)

print("\nAll Picard solves complete.")



-- Picard solve, gaussian: xi=10 nm, D_0=2.26e-20, m_prime=1 --
  eta=0.000  (1/21)
  iter     0  |ΔQ| = 1.400e+02
  eta=0.100  (2/21)
  iter     0  |ΔQ| = 3.041e+01
  eta=0.200  (3/21)
  iter     0  |ΔQ| = 2.536e+00
  eta=0.300  (4/21)
  iter     0  |ΔQ| = 2.524e+00
  eta=0.400  (5/21)
  iter     0  |ΔQ| = 2.516e+00
  eta=0.500  (6/21)
  iter     0  |ΔQ| = 2.508e+00
  eta=0.600  (7/21)
  iter     0  |ΔQ| = 2.499e+00
  eta=0.700  (8/21)
  iter     0  |ΔQ| = 2.492e+00
  eta=0.800  (9/21)
  iter     0  |ΔQ| = 2.484e+00
  eta=0.900  (10/21)
  iter     0  |ΔQ| = 2.476e+00
  eta=1.000  (11/21)
  iter     0  |ΔQ| = 2.468e+00
  eta=1.100  (12/21)
  iter     0  |ΔQ| = 2.460e+00
  eta=1.200  (13/21)
  iter     0  |ΔQ| = 2.453e+00
  eta=1.300  (14/21)
  iter     0  |ΔQ| = 2.445e+00
  eta=1.400  (15/21)
  iter     0  |ΔQ| = 2.438e+00
  eta=1.500  (16/21)
  iter     0  |ΔQ| = 2.431e+00
  eta=1.600  (17/21)
  iter     0  |ΔQ| = 2.424e+00
  eta=1.700  (18/21)
  iter     0  |ΔQ| = 2.417e+00
  eta=1.